# 같은 과제로 다섯 패턴 비교

같은 입력·모델·출력 상한·채점 기준으로 다섯 패턴을 실제 호출해 비교합니다.

## 실행 환경과 비용 상한

최대 18회, 호출당 출력 500토큰으로 제한합니다. 계획 비용은 0.0600달러 미만입니다.

## 코드를 읽기 전에

다섯 함수는 같은 안내문 요구와 모델을 사용하지만 내부 호출 구조가 다릅니다. 각 함수가 시작할 때 호출 기록 위치와 시간을 저장하고, 끝날 때 새로 추가된 성공 응답의 수·토큰·벽시계 시간을 계산합니다. `grade()`는 날짜, 대상, 번호 1·2·3, 문의처, 문자 수라는 공개 항목만 검사해 0~5점을 반환합니다.

같은 점수는 문장 전체 품질이 같다는 뜻이 아닙니다. 채점하지 않은 사실 정확성·문체·독자 이해도는 비교 대상 밖입니다. 또한 한 번의 벽시계 시간은 네트워크 변동을 포함하므로 고정 속도 순위로 읽지 않습니다.

In [1]:
# 이 장은 OpenAI SDK를 직접 배우므로 두 패키지만 고정합니다.
%pip install -q openai==2.29.0 python-dotenv==1.2.3


Note: you may need to restart the kernel to use updated packages.


### 다음 셀을 읽는 방법

`load_dotenv(find_dotenv(usecwd=True))`는 현재 폴더에서 위로 올라가며 `.env`를 찾아 환경 변수에 넣습니다. `AsyncOpenAI(timeout=45.0, max_retries=0)`는 비동기 클라이언트를 만들며 아직 모델을 호출하지 않습니다. `calls`는 이후 성공 응답의 지연과 토큰 수를 모을 빈 목록입니다.


In [2]:
# 비동기 API 호출과 사용량 측정에 필요한 도구를 준비합니다.
import asyncio
import json
import os
import re
import time
from collections import Counter
from dotenv import find_dotenv, load_dotenv
from openai import AsyncOpenAI
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
MODEL = "gpt-5.4-mini"
client = AsyncOpenAI(timeout=45.0, max_retries=0)
calls: list[dict[str, float | int | str]] = []
print("모델:", MODEL, "| API 키 존재: 예")


모델: gpt-5.4-mini | API 키 존재: 예


### 다음 셀을 읽는 방법

`call_json(instructions, prompt, limit)`은 지시문·사용자 입력·최대 출력 토큰을 받는 코루틴입니다. `await client.responses.create()`에서 실제 요청을 기다리고, 응답의 JSON 텍스트를 파이썬 사전으로 바꾸어 반환합니다. `calls`에는 성공 응답의 지연과 입력·출력 토큰만 추가됩니다.


In [3]:
# 비동기 Responses API의 JSON 응답과 사용량을 기록합니다.
async def call_json(instructions: str, prompt: str, limit: int = 220) -> dict:
    started = time.perf_counter()
    response = await client.responses.create(
        model=MODEL,
        instructions=instructions + " 반드시 유효한 JSON 객체만 반환하세요.",
        input="JSON 요청\n" + prompt,
        max_output_tokens=limit,
        reasoning={"effort": "none"},
        text={"format": {"type": "json_object"}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens,
                  "output": response.usage.output_tokens})
    return json.loads(response.output_text)


### 다음 셀을 읽는 방법

`call_answer()`는 `answer` 문자열 하나만 허용하는 JSON Schema를 요청에 넣는 코루틴입니다. 비동기 SDK 응답을 사전으로 바꾼 뒤 값이 문자열이며 비어 있지 않은지 다시 검사하고 사전을 반환합니다. 스키마 통과는 답의 사실 정확성까지 보장하지 않습니다.


In [4]:
# 비동기 응답의 answer 형식을 엄격하게 검사합니다.
async def call_answer(instructions: str, prompt: str, limit: int = 400) -> dict:
    started = time.perf_counter()
    response = await client.responses.create(
        model=MODEL, instructions=instructions, input=prompt,
        max_output_tokens=limit, reasoning={"effort": "none"},
        text={"format": {"type": "json_schema", "name": "answer_payload",
              "schema": {"type": "object", "properties": {"answer": {"type": "string"}},
              "required": ["answer"], "additionalProperties": False}, "strict": True}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens, "output": response.usage.output_tokens})
    data = json.loads(response.output_text)
    if not isinstance(data.get("answer"), str) or not data["answer"].strip():
        raise ValueError("응답의 answer가 비어 있습니다.")
    return data


### 다음 셀을 읽는 방법

`cost_usd(records)`는 앞에서 모은 성공 응답의 입력·출력 토큰을 합산해 이 실습에 고정한 단가를 적용합니다. `score_of(data)`는 평가 점수가 불리언이 아닌 숫자이며 0~100 범위인지 확인한 뒤 정수로 반환합니다. 두 함수 모두 모델을 호출하지 않습니다.


In [5]:
# 반환된 토큰 사용량으로 할인 없는 비용을 추정합니다.
def cost_usd(records: list[dict]) -> float:
    return (sum(int(r["input"]) for r in records) * 0.75
            + sum(int(r["output"]) for r in records) * 4.50) / 1_000_000
def score_of(data: dict) -> int:
    value = data.get("score")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError("평가 응답의 score는 숫자여야 합니다.")
    if not 0 <= value <= 100:
        raise ValueError("평가 응답의 score는 0~100이어야 합니다.")
    return int(value)


## 같은 과제와 공개 채점 기준

In [6]:
# 다섯 패턴에 공통으로 적용할 과제와 채점 기준을 선언합니다.
TASK = "2단계 인증 도입 안내를 120~180자로 작성하세요. 2026년 10월 15일, 전 직원, help@company.test를 포함하고 등록 절차를 1)·2)·3)으로 표시합니다."
def grade(answer: str) -> dict[str, int]:
    checks = {
        "date": int("2026년 10월 15일" in answer),
        "audience": int("전 직원" in answer),
        "three_steps": int(bool(re.search(r"1[.)].+2[.)].+3[.)]", answer, re.S))),
        "contact": int("help@company.test" in answer),
        "length": int(120 <= len(answer) <= 180),
    }
    return {**checks, "score": sum(checks.values())}
def answer_of(data: dict) -> str:
    answer = data.get("answer") if isinstance(data, dict) else None
    return answer.strip() if isinstance(answer, str) else ""
results: dict[str, dict] = {}
print("과제:", TASK)
print("채점: 날짜·대상·3단계·문의처·120~180자, 각 1점")


과제: 2단계 인증 도입 안내를 120~180자로 작성하세요. 2026년 10월 15일, 전 직원, help@company.test를 포함하고 등록 절차를 1)·2)·3)으로 표시합니다.
채점: 날짜·대상·3단계·문의처·120~180자, 각 1점


## 패턴 1~2: 체이닝과 라우팅

In [7]:
# 체이닝과 라우팅을 같은 과제로 실행합니다.
start = len(calls); began = time.perf_counter()
outline = await call_json("안내문 기획자입니다.", TASK + " JSON plan", 250)
chain = await call_answer("안내문 작성자입니다.", f"과제: {TASK}\n기획: {outline}", 500)
results["chaining"] = {"answer": answer_of(chain), "from": start,
                       "latency": time.perf_counter() - began}
start = len(calls); began = time.perf_counter()
route = await call_json("문서를 security, hr, finance 중 분류합니다. JSON category.", TASK, 150)
category = route.get("category")
if category not in {"security", "hr", "finance"}:
    routed = {"answer": "", "status": "invalid_category"}
else:
    routed = await call_answer(f"{category} 전문 작성자입니다.", TASK, 500)
results["routing"] = {"answer": answer_of(routed), "from": start,
                     "latency": time.perf_counter() - began}
print("체이닝·라우팅 완료")


체이닝·라우팅 완료


## 패턴 3: 병렬 섹셔닝

In [8]:
# 세 부분을 동시에 작성한 뒤 실제 내용을 편집자에게 전달합니다.
start = len(calls); began = time.perf_counter()
section_jobs = [
    call_answer("안내문 부분 작성자입니다.", f"{TASK}\n담당: 핵심 사실", 220),
    call_answer("안내문 부분 작성자입니다.", f"{TASK}\n담당: 등록 3단계", 220),
    call_answer("안내문 부분 작성자입니다.", f"{TASK}\n담당: 문의처와 길이", 220),
]
sections = await asyncio.gather(*section_jobs)
assert all(answer_of(item) for item in sections)
merged = " ".join(answer_of(item) for item in sections)
parallel = await call_answer("편집자입니다.", f"과제: {TASK}\n부분: {merged}", 500)
results["parallel"] = {"answer": answer_of(parallel), "from": start,
                       "latency": time.perf_counter() - began}
print("병렬 섹셔닝 완료")


병렬 섹셔닝 완료


## 패턴 4: 동적 오케스트레이터-워커

In [9]:
# 동적 계획에서 나온 작업만 워커와 종합 호출에 사용합니다.
start = len(calls); began = time.perf_counter()
plan = await call_json("오케스트레이터입니다.",
                       TASK + " 필요한 작업 2개만 동적으로 정해 JSON tasks(id,instruction).", 500)
tasks = [x for x in plan.get("tasks", []) if x.get("instruction")][:2]
worker_jobs = [call_answer("독립 워커입니다.",
                         f"과제: {TASK}\n담당: {item['instruction']}", 300)
               for item in tasks]
worker_outputs = await asyncio.gather(*worker_jobs) if worker_jobs else []
orchestrated = await call_answer("종합 작성자입니다.",
    f"과제: {TASK}\n결과: {worker_outputs}", 500)
results["orchestrator_worker"] = {"answer": answer_of(orchestrated), "from": start,
                                  "latency": time.perf_counter() - began}
print("동적 작업 수:", len(tasks))


동적 작업 수: 2


## 패턴 5: 생성자-평가자

In [10]:
# 생성자와 평가자를 실행하고 품질 임계값을 적용합니다.
start = len(calls); began = time.perf_counter()
draft = ""; feedback = "첫 초안"; evaluations = []
for iteration in range(2):
    made = await call_answer("생성자입니다.", f"{TASK}\n피드백: {feedback}", 500)
    draft = answer_of(made)
    judged = await call_json("평가자입니다. JSON score 0~100, feedback.",
                             f"과제: {TASK}\n답: {draft}", 200)
    score = score_of(judged)
    evaluations.append(score)
    feedback = str(judged.get("feedback", ""))
    if score >= 85:
        break
results["generator_evaluator"] = {"answer": draft, "from": start,
                                   "latency": time.perf_counter() - began}
print("평가 점수 원자료:", evaluations)


평가 점수 원자료: [96]


## 호출 수·토큰·지연·품질 비교

In [11]:
# 패턴별 호출 수와 토큰, 지연, 품질을 같은 표로 집계합니다.
rows = []
ordered = list(results)
for index, name in enumerate(ordered):
    begin = int(results[name]["from"])
    end = int(results[ordered[index + 1]]["from"]) if index + 1 < len(ordered) else len(calls)
    records = calls[begin:end]
    answer = str(results[name]["answer"])
    rows.append({"pattern": name, "calls": len(records),
                 "tokens": sum(int(r["input"]) + int(r["output"]) for r in records),
                 "latency_s": round(float(results[name]["latency"]), 3),
                 "quality": grade(answer)["score"], "status": "ok" if answer else "no_answer"})
print("pattern | calls | tokens | latency_s | quality/5 | status")
for row in rows:
    print(" | ".join(str(row[key]) for key in
          ["pattern", "calls", "tokens", "latency_s", "quality", "status"]))
best = max(row["quality"] for row in rows)
print("최고 품질 동점 포함:", [row["pattern"] for row in rows if row["quality"] == best])
print("전체 사용량 기준 추정 비용($):", round(cost_usd(calls), 6))


pattern | calls | tokens | latency_s | quality/5 | status
chaining | 2 | 428 | 3.513 | 5 | ok
routing | 2 | 270 | 2.33 | 4 | ok
parallel | 4 | 884 | 7.008 | 4 | ok
orchestrator_worker | 4 | 1049 | 5.11 | 5 | ok
generator_evaluator | 2 | 420 | 2.765 | 4 | ok
최고 품질 동점 포함: ['chaining', 'orchestrator_worker']
전체 사용량 기준 추정 비용($): 0.006563


## 실험 결과 해석

품질은 공개한 다섯 항목을 충족했는지만 셉니다. 문장 미감이나 조직 정책의 정확성을 평가하지 않습니다. 지연은 이 한 번의 실행에서 측정한 벽시계 시간이라 일반적인 속도 순위가 아닙니다. 최고 점수가 같으면 동점을 유지하며, 빈 답이나 거절은 `no_answer`로 기록합니다. 따라서 화려한 패턴이 언제나 더 높은 점수를 낸다고 결론 내리지 않습니다.

## 다른 상황에 적용하기

패턴 선택표를 다른 과제에 재사용하려면 입력, 출력 제한, 평가 기준을 새 과제에 맞춰 다시 고정해야 합니다. 분류가 분명한 문의에서는 라우팅이 유리할 수 있고, 하위 작업 수가 매번 달라지는 조사에서는 오케스트레이터가 필요할 수 있습니다. 이 절의 숫자를 그대로 옮기는 대신 반복 실행과 실패율을 추가해 구조가 주는 차이를 다시 측정합니다.